# 04_04 Classification - LinearSVC
Train and evaluate LinearSVC.

[COMMAND_SO]
Command 1

[COMMAND_MUC_DICH]
- Muc tieu nghiep vu: Train LinearSVC va visual output de doi chieu voi LR/RF/NB.
- Muc tieu ky thuat: Hien thi metric table va confusion matrix tren test set.

In [1]:
from pathlib import Path
import json
from pyspark.sql import SparkSession
from pyspark.ml.classification import LinearSVC
from pyspark.ml.evaluation import MulticlassClassificationEvaluator
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display
spark=(SparkSession.builder.appName('04_04_lsvc').master('local[2]').config('spark.sql.shuffle.partitions','16').getOrCreate())
spark.sparkContext.setLogLevel('WARN')
PROJECT_ROOT=Path.cwd().resolve().parent if Path.cwd().name=='notebooks' else Path.cwd().resolve()
FEATURE_DIR=PROJECT_ROOT/'data'/'processed'/'features'
MODEL_DIR=PROJECT_ROOT/'models'/'classification'/'linear_svc'
METRIC_DIR=PROJECT_ROOT/'reports'/'model_metrics'
MODEL_DIR.mkdir(parents=True, exist_ok=True)
METRIC_DIR.mkdir(parents=True, exist_ok=True)
train_df=spark.read.parquet(str(FEATURE_DIR/'classification_train')).select('order_id','label','features').dropna()
# --- Oversample minority class (class 1) to handle class imbalance ---
_c0 = train_df.filter(train_df['label'] == 0)
_c1 = train_df.filter(train_df['label'] == 1)
_n0, _n1 = _c0.count(), _c1.count()
print(f'Before oversampling: class_0={_n0}, class_1={_n1}, ratio={_n0/_n1:.2f}')
train_df = _c0.unionAll(_c1.sample(withReplacement=True, fraction=_n0/_n1, seed=42))
print(f'After oversampling: {train_df.count()} rows')
val_df=spark.read.parquet(str(FEATURE_DIR/'classification_val')).select('order_id','label','features').dropna()
test_df=spark.read.parquet(str(FEATURE_DIR/'classification_test')).select('order_id','label','features').dropna()
svc=LinearSVC(featuresCol='features',labelCol='label',regParam=0.05,maxIter=100)
m=svc.fit(train_df)
pred_val=m.transform(val_df)
pred_test=m.transform(test_df)
val_f1=MulticlassClassificationEvaluator(labelCol='label',predictionCol='prediction',metricName='f1').evaluate(pred_val)
val_acc=MulticlassClassificationEvaluator(labelCol='label',predictionCol='prediction',metricName='accuracy').evaluate(pred_val)
test_f1=MulticlassClassificationEvaluator(labelCol='label',predictionCol='prediction',metricName='f1').evaluate(pred_test)
test_acc=MulticlassClassificationEvaluator(labelCol='label',predictionCol='prediction',metricName='accuracy').evaluate(pred_test)
val_precision=MulticlassClassificationEvaluator(labelCol='label',predictionCol='prediction',metricName='weightedPrecision').evaluate(pred_val)
val_recall=MulticlassClassificationEvaluator(labelCol='label',predictionCol='prediction',metricName='weightedRecall').evaluate(pred_val)
test_precision=MulticlassClassificationEvaluator(labelCol='label',predictionCol='prediction',metricName='weightedPrecision').evaluate(pred_test)
test_recall=MulticlassClassificationEvaluator(labelCol='label',predictionCol='prediction',metricName='weightedRecall').evaluate(pred_test)
metrics={'model_family':'classification','model_name':'LinearSVC','val_f1':float(val_f1),'val_accuracy':float(val_acc),'val_precision':float(val_precision),'val_recall':float(val_recall),'f1':float(test_f1),'accuracy':float(test_acc),'precision':float(test_precision),'recall':float(test_recall),'test_f1':float(test_f1),'test_accuracy':float(test_acc),'test_precision':float(test_precision),'test_recall':float(test_recall),'train_rows':train_df.count(),'val_rows':val_df.count(),'test_rows':test_df.count()}
print(metrics)
display(pd.DataFrame([metrics]))
cm_pdf=pred_test.groupBy('label','prediction').count().toPandas()
if not cm_pdf.empty:
    cm_table=cm_pdf.pivot(index='label', columns='prediction', values='count').fillna(0).sort_index().sort_index(axis=1)
    plt.figure(figsize=(6,5))
    sns.heatmap(cm_table, annot=True, fmt='.0f', cmap='Blues')
    plt.title('Confusion Matrix - LinearSVC (Test)')
    plt.xlabel('Prediction')
    plt.ylabel('Label')
    plt.tight_layout()
    plt.show()
m.write().overwrite().save(str(MODEL_DIR))
(METRIC_DIR/'classification_linear_svc.json').write_text(json.dumps(metrics,indent=2),encoding='utf-8')

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/04/06 00:23:25 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


26/04/06 00:23:26 WARN Utils: Service 'SparkUI' could not bind on port 4040. Attempting port 4041.


Before oversampling: class_0=52584, class_1=17025, ratio=3.09


After oversampling: 105187 rows


26/04/06 00:23:32 WARN InstanceBuilder: Failed to load implementation from:dev.ludovic.netlib.blas.JNIBLAS
26/04/06 00:23:32 WARN InstanceBuilder: Failed to load implementation from:dev.ludovic.netlib.blas.VectorBLAS


{'model_family': 'classification', 'model_name': 'LinearSVC', 'val_f1': 0.8569156040830088, 'val_accuracy': 0.8651112898900509, 'val_precision': 0.855536792813949, 'val_recall': 0.8651112898900509, 'f1': 0.8634596296409321, 'accuracy': 0.8712791633145616, 'precision': 0.8614869438322976, 'recall': 0.8712791633145615, 'test_f1': 0.8634596296409321, 'test_accuracy': 0.8712791633145616, 'test_precision': 0.8614869438322976, 'test_recall': 0.8712791633145615, 'train_rows': 105187, 'val_rows': 14916, 'test_rows': 14916}


,model_family,model_name,val_f1,val_accuracy,val_precision,val_recall,f1,accuracy,precision,recall,test_f1,test_accuracy,test_precision,test_recall,train_rows,val_rows,test_rows
0,classification,LinearSVC,0.856916,0.865111,0.855537,0.865111,0.86346,0.871279,0.861487,0.871279,0.86346,0.871279,0.861487,0.871279,105187,14916,14916


/var/folders/07/jh7lmpq57r72h5jdjy3vwy740000gn/T/ipykernel_91651/1538958167.py:52: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


556